# Compare local burst_db outputs vs. a GitHub release

Directly compares the JSON outputs in a local folder (e.g. `test_018/`,
`outputs-0190/`, ...) against the matching assets published on a
[burst_db GitHub release](https://github.com/opera-adt/burst_db/releases) for **BASELINING** purposes.

Local files carry a generation date (and sometimes a date range) in their
name that the release asset doesn't share exactly, so files are paired up
by **family**: the filename with any `YYYY-MM-DD` dates (and `..._to_...`
date ranges) stripped out. Only families present on *both* sides are
compared; anything else is listed as unmatched.

Just change `LOCAL_DIR_NAME` / `RELEASE_TAG` below and re-run to compare a
different local output folder or release.

**Expected differences:** `burst_id_list` counts should stay consistent
between a local run and the release for a given frame — a mismatch there
is a real signal worth investigating. `sensing_time_list` counts, on the
other hand, are expected to *grow* in the local run relative to an older
release, since each local run picks up a newer CSLCs in the CMR survey with
more acquisitions. If you're **reproducing a current release
version** (i.e. `RELEASE_TAG` matches the release your local run is meant
to match), no difference is expected in either `burst_id_list` or
`sensing_time_list` counts — any mismatch there is a real signal.

In [1]:
# --- Parameters: edit these to compare a different folder / release ---
LOCAL_DIR_NAME = "test_018_v2" #"outputs-0190"
RELEASE_TAG = "v0.18.0"
# ------------------------------------------------------------------------

import json
import re
import urllib.request
from pathlib import Path

import pandas as pd

# Assumes the notebook is run with the repo root as the working directory.
REPO_ROOT = Path("../.")
LOCAL_DIR = REPO_ROOT / LOCAL_DIR_NAME
CACHE_DIR = REPO_ROOT / "notebooks" / ".release_cache" / RELEASE_TAG
CACHE_DIR.mkdir(parents=True, exist_ok=True)

RELEASE_BASE = f"https://github.com/opera-adt/burst_db/releases/download/{RELEASE_TAG}"
RELEASE_API = f"https://api.github.com/repos/opera-adt/burst_db/releases/tags/{RELEASE_TAG}"

assert LOCAL_DIR.is_dir(), f"missing local dir: {LOCAL_DIR}"


## Discover and pair up files by "family" (date-stripped name)

In [2]:
DATE_RANGE_RE = re.compile(r"\d{4}-\d{2}-\d{2}_to_\d{4}-\d{2}-\d{2}")
DATE_RE = re.compile(r"\d{4}-\d{2}-\d{2}")


def family_key(filename: str) -> str:
    """Filename with generation dates / date ranges stripped, for pairing local <-> release."""
    name = DATE_RANGE_RE.sub("", filename)
    name = DATE_RE.sub("", name)
    name = re.sub(r"[-_]+", "-", name)
    name = re.sub(r"-+(\.[a-zA-Z0-9.]+)$", r"\1", name)
    return name.strip("-")


# Local candidates: opera-disp-s1-*.json (the files this comparison knows how to read)
local_files = sorted(p.name for p in LOCAL_DIR.glob("opera-disp-s1-*.json"))
local_by_family = {family_key(name): name for name in local_files}

# Release candidates, fetched from the GitHub API for the given tag
with urllib.request.urlopen(RELEASE_API) as resp:
    release_info = json.load(resp)
release_assets = [a["name"] for a in release_info["assets"] if a["name"].endswith(".json")]
release_by_family = {family_key(name): name for name in release_assets}

paired_families = sorted(set(local_by_family) & set(release_by_family))
only_local_families = sorted(set(local_by_family) - set(release_by_family))
only_release_families = sorted(set(release_by_family) - set(local_by_family))

print(f"Local dir:    {LOCAL_DIR}")
print(f"Release tag:  {RELEASE_TAG}")
print()
print(f"Paired families ({len(paired_families)}):")
for fam in paired_families:
    print(f"  {fam!r}: local={local_by_family[fam]!r}  release={release_by_family[fam]!r}")
if only_local_families:
    print(f"\nOnly in local, no release match ({len(only_local_families)}):")
    for fam in only_local_families:
        print(f"  {local_by_family[fam]}")
if only_release_families:
    print(f"\nOnly in release, no local match ({len(only_release_families)}):")
    for fam in only_release_families:
        print(f"  {release_by_family[fam]}")

Local dir:    ../test_018_v2
Release tag:  v0.18.0

Paired families (5):
  'opera-disp-s1-blackout-dates.json': local='opera-disp-s1-blackout-dates-2026-08-03.json'  release='opera-disp-s1-blackout-dates-2026-04-25.json'
  'opera-disp-s1-consistent-burst-ids-no-blackout.json': local='opera-disp-s1-consistent-burst-ids-no-blackout.json'  release='opera-disp-s1-consistent-burst-ids-no-blackout.json'
  'opera-disp-s1-consistent-burst-ids-with-processing-mode.json': local='opera-disp-s1-consistent-burst-ids-with-processing-mode-2026-08-03-2016-07-01_to_2026-04-15.json'  release='opera-disp-s1-consistent-burst-ids-with-processing-mode-2026-04-25-2016-07-01_to_2026-04-15.json'
  'opera-disp-s1-consistent-burst-ids.json': local='opera-disp-s1-consistent-burst-ids-2026-08-03-2016-07-01_to_2026-04-15.json'  release='opera-disp-s1-consistent-burst-ids-2026-04-25-2016-07-01_to_2026-04-15.json'
  'opera-disp-s1-reference-dates.json': local='opera-disp-s1-reference-dates-2026-08-03.json'  release='

## Download release assets and load JSON on both sides

In [3]:
def download(asset_name: str) -> Path:
    dest = CACHE_DIR / asset_name
    if not dest.exists():
        url = f"{RELEASE_BASE}/{asset_name}"
        print(f"Downloading {url} -> {dest}")
        urllib.request.urlretrieve(url, dest)
    return dest


def load_json(path: Path):
    with open(path) as f:
        return json.load(f)


def content_key(doc: dict) -> str:
    """The single top-level key besides 'metadata' holding the comparable content."""
    keys = [k for k in doc if k != "metadata"]
    assert len(keys) == 1, f"expected exactly one non-metadata key, got {keys}"
    return keys[0]


loaded = {}
for fam in paired_families:
    local_path = LOCAL_DIR / local_by_family[fam]
    release_path = download(release_by_family[fam])
    local_doc = load_json(local_path)
    release_doc = load_json(release_path)
    loaded[fam] = {
        "local_path": local_path,
        "release_path": release_path,
        "content_key": content_key(local_doc),
        "local": local_doc,
        "release": release_doc,
    }

## Metadata (expected to differ — dates, filenames, timestamps)

In [4]:
for fam, d in loaded.items():
    print(f"=== {fam} ===")
    print("local metadata:  ", d["local"].get("metadata"))
    print("release metadata:", d["release"].get("metadata"))
    print()

=== opera-disp-s1-blackout-dates.json ===
local metadata:   {'generation_time': '2026-08-03T17:55:31.769814', 'max_default_duration': 240.0, 'input_file': 'opera-region4-snow-analysis.parquet', 'output_file': 'opera-disp-s1-blackout-dates-2026-08-03.json', 'global_blackout_added': {'start': '2025-04-29T19:40:10', 'end': '2025-05-01T19:33:34', 'added_at': '2026-08-03T17:55:31.769825'}}
release metadata: {'generation_time': '2026-04-25T13:02:03.217536', 'max_default_duration': 240.0, 'input_file': 'opera-region4-snow-analysis.parquet', 'output_file': 'opera-disp-s1-blackout-dates-2026-04-25.json', 'global_blackout_added': {'start': '2025-04-29T19:40:10', 'end': '2025-05-01T19:33:34', 'added_at': '2026-04-25T13:02:03.217547'}}

=== opera-disp-s1-consistent-burst-ids-no-blackout.json ===
local metadata:   {'generation_time': '2026-08-03 17:56:20.888076', 'blackout_file': None, 'input_cmr_csv': 'cmr_survey_2016-07-01_to_2026-04-15.csv', 'opera_db_file': 'opera-s1-disp-0.18.0.gpkg', 'generat

## Compare content

For each paired file, diff the dict keyed by burst/frame id (auto-detected
non-metadata key): key-set differences plus per-key value equality.

In [5]:
comparisons = {}
for fam, d in loaded.items():
    key = d["content_key"]
    local_content = d["local"][key]
    release_content = d["release"][key]

    local_keys = set(local_content.keys())
    release_keys = set(release_content.keys())
    common = local_keys & release_keys

    mismatches = {
        k: (local_content[k], release_content[k])
        for k in common
        if local_content[k] != release_content[k]
    }

    comparisons[fam] = {
        "local_keys": local_keys,
        "release_keys": release_keys,
        "only_in_local": local_keys - release_keys,
        "only_in_release": release_keys - local_keys,
        "mismatches": mismatches,
        "identical": local_content == release_content,
    }

    c = comparisons[fam]
    print(f"=== {fam} ===")
    print(f"  local keys:      {len(local_keys)}")
    print(f"  release keys:    {len(release_keys)}")
    print(f"  only in local:   {len(c['only_in_local'])}")
    print(f"  only in release: {len(c['only_in_release'])}")
    print(f"  value mismatches: {len(mismatches)} (out of {len(common)} common keys)")
    print()

=== opera-disp-s1-blackout-dates.json ===
  local keys:      915
  release keys:    915
  only in local:   0
  only in release: 0
  value mismatches: 0 (out of 915 common keys)

=== opera-disp-s1-consistent-burst-ids-no-blackout.json ===
  local keys:      1421
  release keys:    1421
  only in local:   0
  only in release: 0
  value mismatches: 0 (out of 1421 common keys)

=== opera-disp-s1-consistent-burst-ids-with-processing-mode.json ===
  local keys:      1421
  release keys:    1421
  only in local:   0
  only in release: 0
  value mismatches: 0 (out of 1421 common keys)

=== opera-disp-s1-consistent-burst-ids.json ===
  local keys:      1421
  release keys:    1421
  only in local:   0
  only in release: 0
  value mismatches: 0 (out of 1421 common keys)

=== opera-disp-s1-reference-dates.json ===
  local keys:      915
  release keys:    915
  only in local:   0
  only in release: 0
  value mismatches: 0 (out of 915 common keys)



## Mismatched keys as tables

In [6]:
def to_hashable(x):
    if isinstance(x, list):
        return tuple(to_hashable(i) for i in x)
    return x


def diff_list_values(local_list, release_list):
    l = {to_hashable(x) for x in local_list}
    r = {to_hashable(x) for x in release_list}
    return len(l), len(r), len(l - r), len(r - l)


def mismatch_row(key, local_v, release_v):
    row = {"key": key}
    if isinstance(local_v, dict):
        # dict of list-valued fields, e.g. burst_id_list / sensing_time_list
        for field in local_v:
            lc, rc, lo, ro = diff_list_values(local_v[field], release_v[field])
            row[f"{field}_local"] = lc
            row[f"{field}_release"] = rc
            row[f"{field}_only_local"] = lo
            row[f"{field}_only_release"] = ro
    else:
        # plain list value, e.g. blackout intervals / reference dates
        lc, rc, lo, ro = diff_list_values(local_v, release_v)
        row["local_count"] = lc
        row["release_count"] = rc
        row["only_local"] = lo
        row["only_release"] = ro
    return row


mismatch_tables = {}
for fam, comp in comparisons.items():
    rows = [mismatch_row(k, lv, rv) for k, (lv, rv) in comp["mismatches"].items()]
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("key").reset_index(drop=True)
    mismatch_tables[fam] = df

In [7]:
for fam, df in mismatch_tables.items():
    print(f"=== {fam}: {len(df)} mismatched keys ===")
    display(df)

=== opera-disp-s1-blackout-dates.json: 0 mismatched keys ===


""


=== opera-disp-s1-consistent-burst-ids-no-blackout.json: 0 mismatched keys ===


""


=== opera-disp-s1-consistent-burst-ids-with-processing-mode.json: 0 mismatched keys ===


""


=== opera-disp-s1-consistent-burst-ids.json: 0 mismatched keys ===


""


=== opera-disp-s1-reference-dates.json: 0 mismatched keys ===


""


## Map the mismatched frames

`key` in the tables above is the DISP-S1 `frame_id`. Pull frame geometry from
the local `opera-s1-disp-0.18.0.gpkg` (`frames` layer, `fid` == `frame_id`) —
the same static frame database both the local run and the release run were
generated against.

One map per mismatched file, with a toggleable layer per field
(`burst_id_list`, `sensing_time_list`): each frame is colored by
**release count − local count** on a diverging scale — **red = release has
more, blue = local has more** — so the sign is legible at a glance without a
combined score. The colorbar legend sits bottom-left; the layer control
(top-right) switches between fields.

Requires a kernel with `geopandas` + `folium` + `mapclassify` (the `burst_db`
kernel doesn't have these); this was run with the `opera-utils` kernel.

In [8]:
import os
import sys
from pathlib import Path

# conda-forge geopandas builds need PROJ_DATA set before pyproj/geopandas import
# (the proj context caches its search path at import time, not first use)
os.environ.setdefault("PROJ_DATA", str(Path(sys.prefix) / "share" / "proj"))

import geopandas as gpd

mismatched_files = {fam: df for fam, df in mismatch_tables.items() if not df.empty}

frames_gdf = gpd.read_file(
    LOCAL_DIR / "opera-s1-disp-0.18.0.gpkg", layer="frames", fid_as_index=True
)
frames_gdf = frames_gdf.reset_index().rename(columns={"fid": "frame_id"})[["frame_id", "geometry"]]
frames_gdf["frame_id"] = frames_gdf["frame_id"].astype(int)
print(f"loaded {len(frames_gdf):,} frame geometries")

loaded 46,986 frame geometries


In [9]:
import folium
from branca.element import ENV
import branca.colormap as bcm

# branca hardcodes the colorbar legend at 'topright'; move it to bottom-left
# so it doesn't collide with the layer control (also top-right).
_color_scale_src = ENV.loader.get_source(ENV, "color_scale.js")[0]
_color_scale_src = _color_scale_src.replace("position: 'topright'", "position: 'bottomleft'")
bcm.ColorMap._template = ENV.from_string(_color_scale_src)

# Diverging, one colormap per field so the two layers are visually
# distinguishable at a glance: red = release has more, blue/purple = local
# has more, white/light = equal.
FIELD_CMAPS = {
    "burst_id_list": "RdYlBu_r",
    "sensing_time_list": "PuBuGn_r",
}
DEFAULT_CMAP = "RdYlBu_r"


def get_fields(df):
    """(field_name, local_col, release_col) triples for a mismatch table."""
    fields = []
    for c in df.columns:
        if c.endswith("_local") and not c.endswith("_only_local"):
            field = c[: -len("_local")]
            fields.append((field, c, f"{field}_release"))
    return fields


# Human-readable stand-ins for the raw <field>_diff column names, so the
# layer-control checkbox and colorbar caption read as English, not code.
FIELD_LABELS = {
    "burst_id_list": "burst count",
    "sensing_time_list": "sensing time count",
}


def pretty_field(field):
    return FIELD_LABELS.get(field, field.replace("_", " "))


maps = {}
for fam, df in mismatched_files.items():
    gdf = frames_gdf.merge(df.assign(frame_id=df["key"].astype(int)), on="frame_id", how="inner")
    fields = get_fields(df)

    m = None
    for i, (field, local_col, release_col) in enumerate(fields):
        diff_col = f"{field}_diff"
        gdf[diff_col] = gdf[release_col] - gdf[local_col]
        vmax = max(gdf[diff_col].abs().max(), 1)
        cmap = FIELD_CMAPS.get(field, DEFAULT_CMAP)

        explore_kwargs = dict(
            column=diff_col,
            cmap=cmap,
            vmin=-vmax,
            vmax=vmax,
            tooltip=["frame_id", local_col, release_col, diff_col],
            legend=True,
            legend_kwds={
                "caption": (
                    f"{pretty_field(field)} ({cmap}): more in local "
                    "↔ more in release"
                ),
            },
            style_kwds={"weight": 1, "fillOpacity": 0.8},
            name=f"{pretty_field(field)}: release − local",
            show=(i == 0),  # only the first layer starts visible
        )
        if m is None:
            # named TileLayer: geopandas otherwise labels the base layer
            # entry in the layer control with the raw tile URL template
            explore_kwargs["tiles"] = folium.TileLayer(
                tiles="CartoDB positron", attr="CARTO", name="CartoDB Positron",
            )
        else:
            explore_kwargs["m"] = m
        m = gdf.explore(**explore_kwargs)

    folium.LayerControl(collapsed=False).add_to(m)
    maps[fam] = m
    print(f"=== {fam}: {len(gdf)} mismatched frames ({len(fields)} toggleable layers) ===")
    display(m)
    
    m.save(LOCAL_DIR/f"mismatched_frames_{fam}.html")

## Summary

In [10]:
summary_df = pd.DataFrame([
    {
        "file": fam,
        "identical": comp["identical"],
        "common_keys": len(comp["local_keys"] & comp["release_keys"]),
        "value_mismatches": len(comp["mismatches"]),
    }
    for fam, comp in comparisons.items()
])

non_identical = summary_df[~summary_df["identical"]]
if not non_identical.empty:
    print("MISMATCH DETECTED — local output does not match the release:")
    for fam in non_identical["file"]:
        print(f"  {fam}")
    print(
        "\n !!!! Contact PST or ADT leads for verification before publishing this release !!!! "
    )

summary_df

,file,identical,common_keys,value_mismatches
0,opera-disp-s1-blackout-dates.json,True,915,0
1,opera-disp-s1-consistent-burst-ids-no-blackout...,True,1421,0
2,opera-disp-s1-consistent-burst-ids-with-proces...,True,1421,0
3,opera-disp-s1-consistent-burst-ids.json,True,1421,0
4,opera-disp-s1-reference-dates.json,True,915,0


In [11]:
# Save per-file mismatch tables to CSV, next to the local input files
for fam, df in mismatch_tables.items():
    if not df.empty:
        safe_name = re.sub(r"[^A-Za-z0-9]+", "_", fam).strip("_")
        out_csv = LOCAL_DIR / f"mismatch_summary_{safe_name}.csv"
        df.to_csv(out_csv, index=False)
        print(f"wrote {out_csv}")